In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset



In [2]:
class LSTM_DataSet(Dataset):
    def __init__(self, df, window_size, features, targets, wait_id=0):
        self.df = df
        self.window_size = window_size
        self.features = features      # list of feature column names
        self.targets = targets        # list like ["action_id", "pos_x", "pos_y"]
        self.wait_id = wait_id

        # Nested:
        self.X_per_match = []
        self.Y_per_match = []

        # Flattened:
        self.X_flat = []   # each: [T, num_features]
        self.action_flat = []  # each: scalar int
        self.pos_flat = []     # each: [2]
        self.pos_mask_flat = []  # each: bool (should we include in pos loss?)

    def get_match_ids(self):
        return self.df["match_id"].unique().tolist()

    def transform(self):
        match_ids = self.get_match_ids()
        self.X_per_match = []
        self.Y_per_match = []

        for mid in match_ids:
            match_set = self.df[self.df["match_id"] == mid]
            match_X = []
            match_Y = []

            for i in range(len(match_set)):
                start = max(0, i - self.window_size)

                if i == 0:
                    window_frames = []
                else:
                    window_frames = match_set.iloc[start:i][self.features].values.tolist()

                if len(window_frames) < self.window_size:
                    pad_rows = self.window_size - len(window_frames)
                    padding = [[0] * len(self.features)] * pad_rows
                    window_frames = padding + window_frames

                target_row = match_set.iloc[i][self.targets]
                target_list = target_row.values.tolist()  # [action_id, pos_x, pos_y]

                match_X.append(window_frames)
                match_Y.append(target_list)

            self.X_per_match.append(match_X)
            self.Y_per_match.append(match_Y)

        # Now flatten across matches
        self.flatten()

    def flatten(self):
        self.X_flat = []
        self.action_flat = []
        self.pos_flat = []
        self.pos_mask_flat = []

        for match_X, match_Y in zip(self.X_per_match, self.Y_per_match):
            for window_frames, target_list in zip(match_X, match_Y):
                action_id = int(target_list[0])
                pos_x = float(target_list[1])
                pos_y = float(target_list[2])

                self.X_flat.append(window_frames)        # [T, num_features]
                self.action_flat.append(action_id)       # scalar
                self.pos_flat.append([pos_x, pos_y])     # [2]

                # mask: true if we should use this in position loss
                # i.e., not wait and position is valid
                use_pos = (action_id != self.wait_id) and (pos_x != -1) and (pos_y != -1)
                self.pos_mask_flat.append(use_pos)

    def __len__(self):
        return len(self.X_flat)

    def __getitem__(self, idx):
        x = torch.tensor(self.X_flat[idx], dtype=torch.float32)          # [T, F]
        action = torch.tensor(self.action_flat[idx], dtype=torch.long)   # scalar
        pos = torch.tensor(self.pos_flat[idx], dtype=torch.float32)      # [2]
        pos_mask = torch.tensor(self.pos_mask_flat[idx], dtype=torch.bool)  # scalar

        return x, action, pos, pos_mask

In [3]:
full_df = pd.read_csv('full_cleaned_dataset_lstm.csv')
features = [col for col in full_df.columns if col not in ['match_id',"id" ,'action', 'pos_x', 'pos_y']]
targets = ['action', 'pos_x', 'pos_y']

In [4]:
dataset = LSTM_DataSet(full_df, window_size=3, features=features, targets=targets)
dataset.transform()

In [5]:
train_transformed = LSTM_DataSet(pd.read_csv(r"C:\Users\SlayerDz\PycharmProjects\clash-royale-rl-agent\Ai\Behavior_Cloning\train_cleaned_dataset.csv"), window_size=10, features=features, targets=targets)
train_transformed.transform()
test_transformed = LSTM_DataSet(pd.read_csv(r"C:\Users\SlayerDz\PycharmProjects\clash-royale-rl-agent\Ai\Behavior_Cloning\test_cleaned_dataset.csv"), window_size=10, features=features, targets=targets)
test_transformed.transform()

In [6]:
#building the lstm class with 2 heads, one for action and one for positions
class LSTM_BehaviorCloning(nn.Module):
    def __init__(self,input_size, hidden_size,num_layers,num_actions,dropout =0.1):
        super(LSTM_BehaviorCloning, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size,
                            hidden_size=hidden_size,
                            num_layers=num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.shared = nn.Sequential(
            nn.Linear(hidden_size,hidden_size),
            nn.ReLU()
        )

        self.action_head = nn.Linear(hidden_size, num_actions)
        self.pos_head = nn.Linear(hidden_size, 2)

    def forward(self,x):
        lstm_out, _ = self.lstm(x) # [B,T,H]
        last = lstm_out[:,-1,:]
        last_transformed = self.shared(last)

        action_logits = self.action_head(last_transformed)
        pos_logits = self.pos_head(last_transformed)

        return action_logits, pos_logits

In [7]:
#Loading the data for the training loop
from torch.utils.data import DataLoader
train_loaded = DataLoader(train_transformed, batch_size=64, shuffle=True)
test_loaded = DataLoader(test_transformed, batch_size=64, shuffle=False)

In [8]:
import torch.nn.functional as F
def compute_loss(action_logits, pos_pred, target_actions, target_pos, pos_mask, pos_weight=1.0,wait_weight=0.1):
    #we use a weight class to give less importance to the "wait" action in the action loss, since it doesn't have a meaningful position target
    # action weights: 0.1 for wait, 1.0 for others
    num_actions = action_logits.size(1)
    class_weights = torch.ones(num_actions, device=action_logits.device)
    class_weights[0] = wait_weight  # assuming action_id 0 is "wait

    # action classification loss (mean over batch)
    action_loss = F.cross_entropy(action_logits, target_actions,weight=class_weights)

    # position regression loss (mean over valid position samples only)
    if pos_mask.any():
        pos_loss = F.mse_loss(pos_pred[pos_mask], target_pos[pos_mask], reduction="mean")
    else:
        pos_loss = torch.zeros((), device=action_logits.device, dtype=action_logits.dtype)

    total_loss = action_loss + (pos_weight * pos_loss)
    return total_loss, action_loss, pos_loss

In [9]:
import json
import time
from pathlib import Path

def _append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


def run_one_epoch(
    model,
    data_loader,
    device,
    split,                 # "train" or "val"
    epoch_idx,
    optimizer=None,
    pos_weight=1.0,
    log_file=None,         # e.g. "training_metrics.jsonl"
):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss_sum = 0.0
    action_loss_sum = 0.0
    pos_loss_sum = 0.0

    total_samples = 0
    correct_actions = 0

    valid_pos_samples = 0
    valid_pos_coords = 0
    pos_abs_err_sum = 0.0

    t0 = time.time()

    for x, action, pos, pos_mask in data_loader:
        x = x.to(device)
        action = action.to(device)
        pos = pos.to(device)
        pos_mask = pos_mask.to(device)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            action_logits, pos_pred = model(x)
            total_loss, action_loss, pos_loss = compute_loss(
                action_logits=action_logits,
                pos_pred=pos_pred,
                target_actions=action,
                target_pos=pos,
                pos_mask=pos_mask,
                pos_weight=pos_weight,
            )

            if is_train:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        bs = x.size(0)
        total_samples += bs

        total_loss_sum += total_loss.item() * bs
        action_loss_sum += action_loss.item() * bs

        pred_actions = action_logits.argmax(dim=1)
        correct_actions += (pred_actions == action).sum().item()

        if pos_mask.any():
            batch_valid_samples = int(pos_mask.sum().item())
            valid_pos_samples += batch_valid_samples
            pos_loss_sum += pos_loss.item() * batch_valid_samples

            abs_err = (pos_pred[pos_mask] - pos[pos_mask]).abs()  # [valid_samples, 2]
            pos_abs_err_sum += abs_err.sum().item()
            valid_pos_coords += abs_err.numel()

    epoch_record = {
        "event": "epoch",
        "epoch": epoch_idx,
        "split": split,
        "total_loss": total_loss_sum / max(total_samples, 1),
        "action_loss": action_loss_sum / max(total_samples, 1),
        "pos_loss": pos_loss_sum / max(valid_pos_samples, 1),
        "action_acc": correct_actions / max(total_samples, 1),
        "pos_mae": pos_abs_err_sum / max(valid_pos_coords, 1),
        "total_samples": total_samples,
        "valid_pos_samples": valid_pos_samples,
        "valid_pos_coords": valid_pos_coords,
        "correct_actions": correct_actions,
        "epoch_seconds": time.time() - t0,
        "timestamp": time.time(),
    }

    if log_file is not None:
        _append_jsonl(log_file, epoch_record)

    return epoch_record


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    optimizer,
    num_epochs=20,
    pos_weight=1.0,
    log_file="training_metrics.jsonl",
    reset_log_file=True,
):
    log_path = Path(log_file)
    if reset_log_file and log_path.exists():
        log_path.unlink()

    history = {
        "log_file": str(log_path),
        "train": [],
        "val": [],
    }

    for epoch in range(1, num_epochs + 1):
        train_epoch = run_one_epoch(
            model=model,
            data_loader=train_loader,
            device=device,
            split="train",
            epoch_idx=epoch,
            optimizer=optimizer,
            pos_weight=pos_weight,
            log_file=log_path,
        )

        val_epoch = run_one_epoch(
            model=model,
            data_loader=val_loader,
            device=device,
            split="val",
            epoch_idx=epoch,
            optimizer=None,
            pos_weight=pos_weight,
            log_file=log_path,
        )

        history["train"].append(train_epoch)
        history["val"].append(val_epoch)

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_epoch['total_loss']:.4f} train_acc={train_epoch['action_acc']:.4f} train_pos_mae={train_epoch['pos_mae']:.4f} | "
            f"val_loss={val_epoch['total_loss']:.4f} val_acc={val_epoch['action_acc']:.4f} val_pos_mae={val_epoch['pos_mae']:.4f}"
        )

    return history

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTM_BehaviorCloning(
    input_size=len(features),
    hidden_size=128,
    num_layers=2,
    num_actions=13,
    dropout=0.1
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)



In [11]:
history = train_model(
    model=model,
    train_loader=train_loaded,
    val_loader=test_loaded,
    device=device,
    optimizer=optimizer,
    num_epochs=30,
    pos_weight=0.7,
    log_file="training_metrics.jsonl",
)

torch.save(model.state_dict(), "lstm.pth")
print("Model saved to lstm.pth")

Epoch 001 | train_loss=15.6609 train_acc=0.5168 train_pos_mae=3.5805 | val_loss=5.4611 val_acc=0.7087 val_pos_mae=1.6681
Epoch 002 | train_loss=5.7641 train_acc=0.7436 train_pos_mae=1.8502 | val_loss=5.8775 val_acc=0.7087 val_pos_mae=1.8442
Epoch 003 | train_loss=5.7282 train_acc=0.7436 train_pos_mae=1.8530 | val_loss=5.8924 val_acc=0.7087 val_pos_mae=1.8455
Epoch 004 | train_loss=5.7364 train_acc=0.6779 train_pos_mae=1.8524 | val_loss=5.9410 val_acc=0.7087 val_pos_mae=1.8594
Epoch 005 | train_loss=5.8205 train_acc=0.7436 train_pos_mae=1.8609 | val_loss=5.9484 val_acc=0.7087 val_pos_mae=1.8566
Epoch 006 | train_loss=5.5662 train_acc=0.7436 train_pos_mae=1.7974 | val_loss=5.9243 val_acc=0.7087 val_pos_mae=1.8362
Epoch 007 | train_loss=5.1923 train_acc=0.7426 train_pos_mae=1.7032 | val_loss=6.0447 val_acc=0.7087 val_pos_mae=1.8725
Epoch 008 | train_loss=4.9894 train_acc=0.7436 train_pos_mae=1.6452 | val_loss=5.5546 val_acc=0.7087 val_pos_mae=1.7254
Epoch 009 | train_loss=4.6265 train_acc